# Data Cleaning and Preprocessing Script
**Project:** Importing and cleaning a messy e-commerce dataset.

**Tasks Completed:** * Descriptive data exploration
* Handling missing value imputations
* Fixing datatype anomalies
* Cleansing data duplications

In [24]:
import pandas as pd
import numpy as np

# the messy dataset created earlier (or create it if missing)
MESSY_PATH = 'messy_ecommerce_transactions.csv'
if not pd.io.common.file_exists(MESSY_PATH):
    # If the messy file is missing, create a small fallback sample
    data = {
        'Transaction_ID': [f'TXN{i:03d}' for i in range(1, 11)],
        'Product_Name': [
            'Laptop', 'T-Shirt', 'Headphones', 'Smartphone', np.nan,
            'Headphones', 'Watch', 'Jeans', 'Sneakers', 'Tablet'
        ],
        'Price': [
            '$1,200.50', '25.00', '150', '$899.99', np.nan,
            '150', '$200', '45', '$80.50', np.nan
        ],
        'Order_Date': [
            '15-05-2026', '16/05/2026', '17-May-26', '2026-05-18', np.nan,
            '17-May-26', '20-05-2026', '21-05-2026', '22/05/2026', '23-May-26'
        ],
        'Payment_Status': [
            'Success', 'Pending', 'Success', np.nan, 'Failed',
            'Success', 'Success', np.nan, 'Success', 'Pending'
        ]
    }
    pd.DataFrame(data).to_csv(MESSY_PATH, index=False)

# Read the messy CSV (this workspace has a 100-row messy sample already)
messy_df = pd.read_csv(MESSY_PATH)
print('Loaded messy dataset:', MESSY_PATH, '->', messy_df.shape)
messy_df.head(105)

Loaded messy dataset: messy_ecommerce_transactions.csv -> (105, 7)


,Transaction_ID,Product_Name,Price,Order_Date,Quantity,Customer_Email,Payment_Status
0,TXN0001,USB Cable,$287.68,20-05-2026,NaN,customer36@gmail.com,Success
1,TXN0002,RAM,"$1,182.64",2026-02-10,5.0,customer7@email.com,SUCCESS
2,TXN0003,USB Cable,"$1,462.17",07-04-2026,5.0,customer3@gmail.com,Failure
3,TXN0004,Headphones,"$1,734",12-Jun-26,4.0,customer39@gmail.com,pending
4,TXN0005,Mouse,$765.12,15/01/2026,1.0,customer14@outlook.com,pending
...,...,...,...,...,...,...,...
100,TXN0006,Cooling Pad,287.86,04/20/2026,5.0,customer49@gmail.com,Succeeded
101,TXN0070,RAM,"$1,456",05/06/2026,5.0,customer16@email.com,Pending
102,TXN0083,Tablet,$586.00,NaN,2.0,customer5@outlook.com,SUCCESS
103,TXN0100,Laptop,$946,06-03-2026,10.0,customer9@outlook.com,success


In [27]:
# Cleaning pipeline that mirrors the cleaned_ecommerce_transactions.csv produced earlier
def clean_transactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Trim whitespace in string columns
    df['Product_Name'] = df['Product_Name'].astype(str).str.strip()
    df['Product_Name'].replace({'nan': pd.NA, '': pd.NA})

    # Normalize price: remove currency symbols and thousands separators, coerce to numeric
    df['Price'] = df['Price'].astype(str).str.replace(r'[\$,]', '', regex=True).str.strip()
    df['Price'].replace({'': pd.NA, 'nan': pd.NA})
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

    # Parse dates (supports multiple formats)
    df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce', dayfirst=True)

    # Normalize payment status to canonical values
    def map_status(x):
        if pd.isna(x):
            return pd.NA
        s = str(x).strip().lower()
        if s in ('success','succeeded','completed','paid','successfull'):
            return 'Success'
        if s in ('pending',):
            return 'Pending'
        if s in ('failed','failed.','failure'):
            return 'Failed'
        return pd.NA

    df['Payment_Status'] = df['Payment_Status'].apply(map_status)

    # Fill missing product names
    df['Product_Name'] = df['Product_Name'].fillna('Unknown')

    # Fill missing prices: use median when available, else 0.00
    if df['Price'].notna().any():
        median_price = df['Price'].median()
    else:
        median_price = 0.0
    df['Price'] = df['Price'].fillna(median_price)

    # Drop duplicate Transaction_IDs, keep first occurrence
    if 'Transaction_ID' in df.columns:
        df = df.drop_duplicates(subset=['Transaction_ID']).reset_index(drop=True)

    # Standardize formats: Price to 2 decimals, Order_Date to ISO string
    df['Price'] = df['Price'].round(2)
    df['Order_Date'] = df['Order_Date'].dt.strftime('%Y-%m-%d')

    # Fill remaining missing payment status as 'Pending' (business decision)
    df['Payment_Status'] = df['Payment_Status'].fillna('Pending')

    return df

# Apply cleaning
cleaned_df = clean_transactions(messy_df)

# Save cleaned output
CLEANED_PATH = 'cleaned_ecommerce_transactions.csv'
cleaned_df.to_csv(CLEANED_PATH, index=False)
print('Saved cleaned dataset ->', CLEANED_PATH)

# Quick validation summary
print('\nValidation summary:')
print('Original rows:', len(messy_df))
print('Cleaned rows:', len(cleaned_df))
print('Duplicate Transaction_IDs removed:', len(messy_df) - len(cleaned_df))
print('\nMissing values per column in cleaned data:')
print(cleaned_df.isna().sum())

# Display first rows of the cleaned set
cleaned_df.head(105)

Saved cleaned dataset -> cleaned_ecommerce_transactions.csv

Validation summary:
Original rows: 105
Cleaned rows: 100
Duplicate Transaction_IDs removed: 5

Missing values per column in cleaned data:
Transaction_ID     0
Product_Name       0
Price              0
Order_Date        83
Quantity           3
Customer_Email     2
Payment_Status     0
dtype: int64


,Transaction_ID,Product_Name,Price,Order_Date,Quantity,Customer_Email,Payment_Status
0,TXN0001,USB Cable,287.68,2026-05-20,NaN,customer36@gmail.com,Success
1,TXN0002,RAM,1182.64,NaN,5.0,customer7@email.com,Success
2,TXN0003,USB Cable,1462.17,2026-04-07,5.0,customer3@gmail.com,Failed
3,TXN0004,Headphones,1734.00,NaN,4.0,customer39@gmail.com,Pending
4,TXN0005,Mouse,765.12,NaN,1.0,customer14@outlook.com,Pending
...,...,...,...,...,...,...,...
95,TXN0096,Tablet,192.51,2026-03-31,3.0,customer24@gmail.com,Failed
96,TXN0097,Watch,1303.42,NaN,7.0,customer6@gmail.com,Success
97,TXN0098,nan,308.00,NaN,10.0,customer5@email.com,Success
98,TXN0099,Laptop,1154.34,NaN,10.0,customer5@outlook.com,Success
